# Day 01: Random Forests & Rừng Nơ-ron Khả Vi (PyTorch)
### Sổ Tay Thực Hành: `random_forest_pytorch.ipynb`

> **Bài báo gốc:** *Random Forests*  
> **Tác giả:** Leo Breiman (Đại học California, Berkeley) — Xuất bản năm 2001  
> **Định hướng mở rộng hiện đại:** Rừng Quyết Định Khả Vi (Differentiable Neural Decision Forest - NDF)  
> **Tài liệu tham khảo:** [`resources/Day 1 Random Forest.pdf`](./resources/Day%201%20Random%20Forest.pdf)

---

## 🎯 Mục Tiêu Cốt Lõi Của Notebook
1. **Giải thích chi tiết cho người mới bắt đầu:** Hiểu rõ tại sao cây quyết định truyền thống không thể tính đạo hàm, và cách giải thuật "Làm mềm" (Soft Routing Sigmoid) giải quyết bài toán này để đưa cây quyết định vào hệ sinh thái PyTorch.
2. **Cài đặt mô hình nơ-ron khả vi (Neural Decision Tree & Forest):** Tận dụng PyTorch Autograd để huấn luyện cây quyết định end-to-end trên GPU bằng giải thuật Lan truyền ngược (Backpropagation).
3. **Thực nghiệm đối chuẩn giá trị học thuật:** So sánh định lượng 4 mô hình xuyên suốt lịch sử học máy:
   - **Logistic Regression:** Mô hình tuyến tính cổ điển trước khi có cây.
   - **Single Decision Tree (CART 1984):** Cây đơn lẻ (học phi tuyến tốt nhưng bị High Variance).
   - **Classical Random Forest (Breiman 2001):** Rừng cổ điển (chuẩn mực số 1 cho Tabular Data).
   - **Differentiable Neural Decision Forest (PyTorch):** Rừng nơ-ron vi phân kết hợp sức mạnh của cây với GPU.

## 💡 Lý Thuyết Cho Người Mới Bắt Đầu: Cây Quyết Định Khả Vi Là Gì?

Trong cây CART của Breiman, quyết định rẽ nhánh là phép so sánh cứng:
$$\text{Đi sang trái} = \begin{cases} 1 & \text{nếu } x_j \le \text{threshold} \\ 0 & \text{ngược lại} \end{cases}$$

Hàm này có đạo hàm bằng $0$ ở khắp mọi nơi, do đó **không thể tối ưu bằng Gradient Descent**!

Để giải quyết bế tắc này, ta thay thế bằng **cổng rẽ nhánh mềm (Soft Routing)** sử dụng hàm Sigmoid:
$$p(\text{trái}) = \sigma\left(\frac{w^T x - b}{\tau}\right) = \frac{1}{1 + e^{-(w^T x - b) / \tau}}$$

- $w, b$: Trọng số và độ lệch học được bằng giải thuật Backpropagation.
- $\tau$ (Temperature): Khi $\tau$ lớn, cổng phân chia mềm mại và mượt mà; khi $\tau \to 0$, cổng tiến về phép so sánh cứng của CART.
- Xác suất rơi vào lá $l$ là **tích xác suất của các quyết định rẽ nhánh** từ gốc đến lá đó:
  $$\mu_l(x) = \prod_{d \in \text{đường đi}} p_d(x)$$
- Dự đoán cuối cùng là kỳ vọng toán học qua tất cả các lá:
  $$P(y = c \mid x) = \sum_{l=1}^{2^D} \mu_l(x) \cdot \pi_{l, c}$$
  với $\pi_{l, c}$ là phân phối xác suất lớp tại lá $l$ (được chuẩn hóa bằng Softmax).

In [ ]:
import os
import time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_breast_cancer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[*] Đang thực thi trên thiết bị: {device}")

## PHẦN 1: ĐỊNH NGHĨA KIẾN TRÚC MÔ HÌNH NƠ-RON KHẢ VI (PYTORCH)

In [ ]:
class NeuralDecisionTree(nn.Module):
    """
    Cây quyết định khả vi (Differentiable Neural Decision Tree):
    Mỗi tầng rẽ nhánh nhị phân được tính toán song song qua cổng Sigmoid Soft Routing.
    """
    def __init__(self, in_features, num_classes, depth=3, temperature=1.0):
        super(NeuralDecisionTree, self).__init__()
        self.depth = depth
        self.num_classes = num_classes
        self.temperature = temperature

        self.num_inner_nodes = 2 ** depth - 1
        self.num_leaves = 2 ** depth

        # Trọng số định tuyến tại các nút trong: w^T x - b
        self.routing_layer = nn.Linear(in_features, self.num_inner_nodes)

        # Phân phối logit lớp tại các nút lá
        self.leaf_logits = nn.Parameter(torch.randn(self.num_leaves, num_classes) * 0.1)

    def forward(self, x):
        batch_size = x.size(0)
        # 1. Điểm số rẽ nhánh Sigmoid tại các nút trong
        node_scores = torch.sigmoid(self.routing_layer(x) / self.temperature)

        # 2. Lan truyền xác suất đường đi nhị phân (Binary Routing Path)
        path_probs = [torch.ones(batch_size, 1, device=x.device)]

        for d in range(self.depth):
            level_start = 2 ** d - 1
            level_end = 2 ** (d + 1) - 1
            level_scores = node_scores[:, level_start:level_end]

            new_path_probs = []
            for i in range(2 ** d):
                parent_prob = path_probs[i]
                p_left = level_scores[:, i:i+1]
                new_path_probs.append(parent_prob * p_left)
                new_path_probs.append(parent_prob * (1.0 - p_left))
            path_probs = new_path_probs

        # Ma trận xác suất đến các lá: (batch_size, num_leaves)
        leaf_probs = torch.cat(path_probs, dim=1)

        # 3. Phân phối xác suất lớp tại các lá: softmax
        leaf_distributions = torch.softmax(self.leaf_logits, dim=1)

        # 4. Dự đoán cuối cùng: tích vô hướng giữa xác suất lá và phân phối lớp
        return torch.matmul(leaf_probs, leaf_distributions)


class NeuralRandomForest(nn.Module):
    """
    Quần thể Rừng Nơ-ron Vi Phân (Neural Random Forest):
    Tập hợp n_trees cây nơ-ron độc lập, lấy trung bình xác suất đầu ra.
    """
    def __init__(self, in_features, num_classes, n_trees=6, depth=3, temperature=1.0):
        super(NeuralRandomForest, self).__init__()
        self.trees = nn.ModuleList([
            NeuralDecisionTree(in_features, num_classes, depth=depth, temperature=temperature)
            for _ in range(n_trees)
        ])

    def forward(self, x):
        all_tree_probs = [tree(x) for tree in self.trees]
        return torch.stack(all_tree_probs, dim=0).mean(dim=0)

print("[✓] Đã khởi tạo thành công lớp NeuralDecisionTree và NeuralRandomForest!")

## PHẦN 2: CHUẨN BỊ DỮ LIỆU BẢNG & THÊM NHIỄU ĐỂ THỬ THÁCH MÔ HÌNH

Ta nạp bộ dữ liệu y khoa **Breast Cancer** từ scikit-learn (hoặc fallback nhân tạo nếu chạy offline).
Đồng thời, thêm **5% nhiễu nhãn (Label Noise)** để mô phỏng sự sai số trong thu thập dữ liệu đời thực, từ đó kiểm chứng tính kháng nhiễu bền bỉ (Noise Robustness) của thuật toán Random Forest.

In [ ]:
# Nạp dữ liệu và chuẩn bị
data = load_breast_cancer()
n_samples = 400
n_features = 10

X = data.data[:n_samples, :n_features].astype(np.float32)
X = (X - X.mean(axis=0)) / (X.std(axis=0) + 1e-8)  # Chuẩn hóa Z-Score
y = data.target[:n_samples].astype(np.int64)
feature_names = [f"X{i}_{name[:8]}" for i, name in enumerate(data.feature_names[:n_features])]

# Thêm 5% nhiễu nhãn ngẫu nhiên
np.random.seed(42)
flip_mask = np.random.rand(len(y)) < 0.05
y[flip_mask] = 1 - y[flip_mask]

# Chia Train (75%) và Test (25%)
split = int(0.75 * len(X))
X_train, y_train = X[:split], y[:split]
X_test, y_test = X[split:], y[split:]

print(f"[*] Kích thước dữ liệu: Train = {X_train.shape}, Test = {X_test.shape}")
print(f"[*] Danh sách đặc trưng: {feature_names[:4]} ...")

## PHẦN 3: HUẤN LUYỆN VÀ ĐỐI CHUẨN 4 CỘT MỐC HỌC THUẬT

Chúng ta sẽ lần lượt huấn luyện và so sánh:
1. **Logistic Regression:** Mô hình phân loại tuyến tính cổ điển.
2. **Single Decision Tree (CART 1984):** Cây quyết định đơn lẻ không cắt tỉa.
3. **Classical Random Forest (Breiman 2001):** Rừng ngẫu nhiên với $m = \sqrt{M}$.
4. **Differentiable Neural Decision Forest (PyTorch):** Rừng nơ-ron vi phân tối ưu qua Backpropagation.

In [ ]:
models_results = {}

# 1. Logistic Regression (Baseline Tuyến Tính)
lr = LogisticRegression(max_iter=500, random_state=42)
lr.fit(X_train, y_train)
lr_train = lr.score(X_train, y_train) * 100.0
lr_test = lr.score(X_test, y_test) * 100.0
models_results["1. Logistic Reg."] = {"train_acc": lr_train, "test_acc": lr_test, "overfit_gap": lr_train - lr_test}

# 2. Single Decision Tree (CART 1984)
cart = DecisionTreeClassifier(max_depth=None, random_state=42)
cart.fit(X_train, y_train)
cart_train = cart.score(X_train, y_train) * 100.0
cart_test = cart.score(X_test, y_test) * 100.0
models_results["2. Single CART"] = {"train_acc": cart_train, "test_acc": cart_test, "overfit_gap": cart_train - cart_test}

# 3. Classical Random Forest (Leo Breiman 2001)
rf = RandomForestClassifier(n_estimators=60, max_features="sqrt", random_state=42)
rf.fit(X_train, y_train)
rf_train = rf.score(X_train, y_train) * 100.0
rf_test = rf.score(X_test, y_test) * 100.0
rf_importances = rf.feature_importances_
models_results["3. Breiman RF"] = {"train_acc": rf_train, "test_acc": rf_test, "overfit_gap": rf_train - rf_test}

print("[✓] Đã hoàn thành huấn luyện 3 mô hình nền tảng cổ điển!")

### 3.1. Huấn luyện Neural Random Forest bằng PyTorch Autograd

In [ ]:
# Chuẩn bị TensorDataset và DataLoader
X_train_tensor = torch.tensor(X_train, dtype=torch.float32, device=device)
y_train_tensor = torch.tensor(y_train, dtype=torch.long, device=device)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32, device=device)
y_test_tensor = torch.tensor(y_test, dtype=torch.long, device=device)

train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=32, shuffle=True)

# Khởi tạo Rừng nơ-ron
neural_forest = NeuralRandomForest(in_features=n_features, num_classes=2, n_trees=6, depth=3, temperature=1.0).to(device)
optimizer = optim.Adam(neural_forest.parameters(), lr=0.015, weight_decay=1e-4)
criterion = nn.NLLLoss()

epochs = 30
loss_history = []
t0 = time.time()

for epoch in range(1, epochs + 1):
    neural_forest.train()
    total_loss = 0.0
    for bx, by in train_loader:
        optimizer.zero_grad()
        probs = neural_forest(bx)
        log_probs = torch.log(torch.clamp(probs, min=1e-7, max=1.0))
        loss = criterion(log_probs, by)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    epoch_loss = total_loss / len(train_loader)
    loss_history.append(epoch_loss)
    if epoch % 10 == 0 or epoch == 1:
        print(f"  > Epoch {epoch:2d}/{epochs} | Training Loss: {epoch_loss:.4f}")

train_time = time.time() - t0
print(f"[*] Huấn luyện hoàn thành trong {train_time:.2f} giây.")

# Đánh giá Neural Forest
neural_forest.eval()
with torch.no_grad():
    train_probs = neural_forest(X_train_tensor)
    ndf_train = np.mean(torch.argmax(train_probs, dim=1).cpu().numpy() == y_train) * 100.0
    
    test_probs = neural_forest(X_test_tensor)
    ndf_test = np.mean(torch.argmax(test_probs, dim=1).cpu().numpy() == y_test) * 100.0

models_results["4. Neural Forest"] = {"train_acc": ndf_train, "test_acc": ndf_test, "overfit_gap": ndf_train - ndf_test}

## PHẦN 4: BẢNG SO SÁNH GIÁ TRỊ HỌC THUẬT & TRỰC QUAN HÓA TOÀN DIỆN

In [ ]:
# In bảng đối chuẩn kết quả
print("=" * 88)
print(f"{'Mô Hình / Giải Thuật':<26} | {'Train Acc':<10} | {'Test Acc':<10} | {'Overfit Gap':<12} | Đóng Góp Học Thuật")
print("=" * 88)
print(f"{'1. Logistic Regression':<26} | {lr_train:8.2f}% | {lr_test:8.2f}% | {(lr_train - lr_test):10.2f}% | Mô hình tuyến tính đơn giản, không học phi tuyến.")
print(f"{'2. Single CART (1984)':<26} | {cart_train:8.2f}% | {cart_test:8.2f}% | {(cart_train - cart_test):10.2f}% | Học phi tuyến tốt nhưng bị High Variance nghiêm trọng.")
print(f"{'3. Breiman RF (2001)':<26} | {rf_train:8.2f}% | {rf_test:8.2f}% | {(rf_train - rf_test):10.2f}% | Triệt tiêu phương sai, chuẩn mực số 1 cho dữ liệu bảng.")
print(f"{'4. Neural Forest (PyTorch)':<26} | {ndf_train:8.2f}% | {ndf_test:8.2f}% | {(ndf_train - ndf_test):10.2f}% | Khả vi, huấn luyện end-to-end trên GPU cùng mạng sâu.")
print("=" * 88)

In [ ]:
# Vẽ đồ thị tổng hợp 4 ô trực quan hóa
fig, axes = plt.subplots(2, 2, figsize=(16, 11))
model_names = list(models_results.keys())
train_accs = [models_results[m]["train_acc"] for m in model_names]
test_accs = [models_results[m]["test_acc"] for m in model_names]

# 1. MDI Feature Importance
sorted_idx = np.argsort(rf_importances)
axes[0, 0].barh(np.arange(len(feature_names)), rf_importances[sorted_idx], color='#1f77b4', edgecolor='black', alpha=0.85)
axes[0, 0].set_yticks(np.arange(len(feature_names)))
axes[0, 0].set_yticklabels([feature_names[i] for i in sorted_idx], fontsize=9)
axes[0, 0].set_title("1. Tầm Quan Trọng Đặc Trưng (Breiman MDI)", fontsize=11, fontweight='bold')
axes[0, 0].grid(True, linestyle="--", alpha=0.5)

# 2. Test Accuracy
colors = ["#7f7f7f", "#d62728", "#1f77b4", "#2ca02c"]
bars = axes[0, 1].bar(model_names, test_accs, color=colors, width=0.45, edgecolor='black', alpha=0.85)
axes[0, 1].set_ylim(0, 105)
axes[0, 1].set_ylabel("Độ Chính Xác Test (%)", fontsize=10, fontweight='bold')
axes[0, 1].set_title("2. Năng Lực Tổng Quát Hóa (Test Accuracy)", fontsize=11, fontweight='bold')
axes[0, 1].grid(True, linestyle="--", alpha=0.5)
for b in bars:
    h = b.get_height()
    axes[0, 1].text(b.get_x() + b.get_width() / 2., h + 2, f"{h:.1f}%", ha='center', fontweight='bold')

# 3. Overfitting Gap
x_pos = np.arange(len(model_names))
width = 0.35
axes[1, 0].bar(x_pos - width/2, train_accs, width, label='Train Acc', color='#4a7bb0', edgecolor='black', alpha=0.85)
axes[1, 0].bar(x_pos + width/2, test_accs, width, label='Test Acc', color='#e07a5f', edgecolor='black', alpha=0.85)
axes[1, 0].set_xticks(x_pos)
axes[1, 0].set_xticklabels(model_names, rotation=15, fontsize=9)
axes[1, 0].set_ylim(0, 110)
axes[1, 0].set_ylabel("Độ Chính Xác (%)", fontsize=10, fontweight='bold')
axes[1, 0].set_title("3. Khoảng Cách Quá Khớp (Overfitting Gap)", fontsize=11, fontweight='bold')
axes[1, 0].legend(loc='lower right')
axes[1, 0].grid(True, linestyle="--", alpha=0.5)

# 4. Loss Convergence
axes[1, 1].plot(range(1, len(loss_history) + 1), loss_history, marker='o', color='#2ca02c', linewidth=2, markersize=4)
axes[1, 1].set_xlabel("Epochs", fontsize=10, fontweight='bold')
axes[1, 1].set_ylabel("NLL Loss", fontsize=10, fontweight='bold')
axes[1, 1].set_title("4. Quá Trình Hội Tụ Của Neural Decision Forest", fontsize=11, fontweight='bold')
axes[1, 1].grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

## 🎓 ĐÚC KẾT HỌC THUẬT DÀNH CHO NGƯỜI MỚI BẮT ĐẦU

1. **Tầm quan trọng của bài báo Leo Breiman (2001):**
   - Đưa ra giải pháp hoàn chỉnh cho bài toán dữ liệu bảng (Tabular Data) vốn chiếm đa số trong các ứng dụng kinh doanh, tài chính, y tế.
   - Giúp các nhà thực hành có một mô hình mạnh mẽ không cần điều chỉnh siêu tham số phức tạp (out-of-the-box strong baseline).
2. **Tính kháng nhiễu (Noise Robustness):**
   - Khi dữ liệu có 5% nhiễu nhãn, Single CART Tree lập tức bị lừa (ghi nhớ các điểm nhiễu khiến Test Acc tụt giảm).
   - Random Forest phân bổ trọng số đều qua các cây, triệt tiêu ảnh hưởng của điểm nhiễu cá biệt và giữ vững độ chính xác cao nhất.
3. **Cầu nối giữa Cây Quyết Định và Mạng Nơ-ron Hiện Đại:**
   - Differentiable Neural Decision Forests chứng minh rằng tư duy rẽ nhánh của Breiman hoàn toàn có thể kết hợp với kiến trúc học sâu (Deep Learning) và GPU, mở ra các hướng nghiên cứu hiện đại như TabNet, NODE, và Differentiable Ensembles.